# Control-plane cost experiment (Colab)
Runs the learned agents under controller-query, signalling and flow-installation costs (reviewer comments R2.2 and R3-M2).

**Before you start:** upload `IoMT_Offloading_Reproducibility_Package.zip` to *My Drive*. Use a plain **CPU** runtime (Runtime > Change runtime type > CPU); a GPU does not help.

Everything is written to Google Drive and every command is **resumable**: if the session disconnects, reconnect, run cells 1-2 and start the block again; finished runs are skipped.

| Block | What | Runs | About (2 cores) |
|---|---|---|---|
| B | `dueling_ddqn` in low / mid / high scenarios, N = 20, 60, 10 seeds | 60 | 2.5 h |
| C | `device_local` (strictly device-local agent): ideal + 3 scenarios | 80 | 3.5 h |
| D | optional: `standard_dqn` in 3 scenarios | 60 | 2.5 h |

Do not run two blocks at the same time on 2 cores.

## 1. Mount Drive and unzip (first time only)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PKG = "/content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package"
import os
if not os.path.isdir(PKG):
    !cd /content/drive/MyDrive && unzip -q -o IoMT_Offloading_Reproducibility_Package.zip
print(os.path.isdir(PKG))
!nproc

## 2. Check packages and the extended simulator

In [ ]:
!python -c "import torch, numpy, scipy, pandas, matplotlib; print('torch', torch.__version__)"
# quick smoke test (1 minute): 2 seeds, tiny training, low scenario; writes to /content, not Drive
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && python run_v2.py --suite main --arms dueling_ddqn --N 20 --seeds 2 --episodes 5 --n-test 300 --jobs 2 --ctrl low --out /content/smoke | tail -n 8

## 3. Set the number of parallel jobs
Use the number printed by `nproc` (2 on free Colab).

In [ ]:
JOBS = 2

## Block B (required): proposed agent in each scenario

In [ ]:
import subprocess
cmd = '''cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && nohup bash -c '
for sc in low mid high; do
  python run_v2.py --suite main --arms dueling_ddqn --N 20,60 --seeds 10 --jobs __JOBS__ --ctrl $sc --out ../raw/ctrl_${sc}_learned
done' > ../blockB.log 2>&1 &'''
subprocess.Popen(cmd.replace('__JOBS__', str(JOBS)), shell=True)
print("Block B started in the background. Check progress with the cell below.")

### Progress of block B (re-run any time)

In [ ]:
!tail -n 3 /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/blockB.log
!wc -l /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw/ctrl_*_learned/per_seed.csv   # 21 lines per folder = finished

## Block C (recommended): strictly device-local agent
Start it only after block B has finished.

In [ ]:
cmd = '''cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && nohup bash -c '
python run_v2.py --suite main --arms device_local --N 20,60 --seeds 10 --jobs __JOBS__ --ctrl none --out ../raw/ctrl_none_device_local
for sc in low mid high; do
  python run_v2.py --suite main --arms device_local --N 20,60 --seeds 10 --jobs __JOBS__ --ctrl $sc --out ../raw/ctrl_${sc}_device_local
done' > ../blockC.log 2>&1 &'''
subprocess.Popen(cmd.replace('__JOBS__', str(JOBS)), shell=True)
print("Block C started in the background.")

### Progress of block C

In [ ]:
!tail -n 3 /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/blockC.log
!wc -l /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/raw/ctrl_*_device_local/per_seed.csv   # 21 lines per folder = finished

## Block D (optional): Standard DQN in the three scenarios

In [ ]:
cmd = '''cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package/code && nohup bash -c '
for sc in low mid high; do
  python run_v2.py --suite main --arms standard_dqn --N 20,60 --seeds 10 --jobs __JOBS__ --ctrl $sc --out ../raw/ctrl_${sc}_standard
done' > ../blockD.log 2>&1 &'''
subprocess.Popen(cmd.replace('__JOBS__', str(JOBS)), shell=True)
print("Block D started in the background.")

## Analysis (after the blocks you ran have finished)
Writes `results/ctrlplane_*.csv` and `figures/Fig_ctrlplane_*.png`.

In [ ]:
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package && python analysis/ctrlplane_analysis.py | tail -n 40

## Collect the results
Zips the new raw folders and result files so you can download them or send them to me.

In [ ]:
!cd /content/drive/MyDrive/IoMT_Offloading_Reproducibility_Package && zip -qr /content/drive/MyDrive/ctrlplane_results.zip raw/ctrl_* results/ctrlplane_* figures/Fig_ctrlplane_* blockB.log blockC.log blockD.log 2>/dev/null; ls -la /content/drive/MyDrive/ctrlplane_results.zip
from google.colab import files
files.download('/content/drive/MyDrive/ctrlplane_results.zip')